In [54]:
import pandas as pd
import numpy as np
import joblib
import os
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# =====================================================
# 1. LOAD DATASET
# =====================================================

df = pd.read_excel("../datasets/carrot_complete_growth_dataset.xlsx")

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

# Remove old avg column if exists
if "Carrot_Avg_Price" in df.columns:
    df = df.drop(columns=["Carrot_Avg_Price"])

# Create numeric average
df["Carrot_Avg_Price"] = (
    df["Carrot_Min_Price"] + df["Carrot_Max_Price"]
) / 2


# =====================================================
# 2. FEATURE ENGINEERING FUNCTION
# =====================================================

def create_features(data):
    df_feat = data.copy()

    # Short-term lag features
    for lag in [1, 2, 3, 7]:
        df_feat[f'avg_lag_{lag}'] = df_feat['Carrot_Avg_Price'].shift(lag)
        df_feat[f'min_lag_{lag}'] = df_feat['Carrot_Min_Price'].shift(lag)
        df_feat[f'max_lag_{lag}'] = df_feat['Carrot_Max_Price'].shift(lag)

    # Rolling features
    df_feat['avg_rolling_mean_7'] = df_feat['Carrot_Avg_Price'].rolling(7).mean()
    df_feat['avg_rolling_std_7'] = df_feat['Carrot_Avg_Price'].rolling(7).std()

    # Date feature
    df_feat['day_of_week'] = df_feat['Date'].dt.dayofweek

    return df_feat


# Apply feature engineering
df = create_features(df)
df = df.dropna().reset_index(drop=True)


# =====================================================
# 3. TRAIN / VALIDATION SPLIT
# =====================================================

train = df[df['Date'] < '2024-01-01']
val = df[df['Date'] >= '2024-01-01']

feature_cols = train.drop(columns=[
    'Date',
    'Carrot_Avg_Price',
    'Carrot_Min_Price',
    'Carrot_Max_Price'
]).columns

X_train = train[feature_cols]
X_val = val[feature_cols]

y_train_min = train['Carrot_Min_Price']
y_train_max = train['Carrot_Max_Price']

y_val_min = val['Carrot_Min_Price']
y_val_max = val['Carrot_Max_Price']


# =====================================================
# 4. TRAIN MODELS
# =====================================================

model_min = XGBRegressor(
    n_estimators=800,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model_max = XGBRegressor(
    n_estimators=800,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model_min.fit(X_train, y_train_min)
model_max.fit(X_train, y_train_max)


# =====================================================
# 5. VALIDATION PERFORMANCE
# =====================================================

pred_min = model_min.predict(X_val)
pred_max = model_max.predict(X_val)

print("MIN PRICE VALIDATION")
print("MAE :", mean_absolute_error(y_val_min, pred_min))
print("RMSE:", np.sqrt(mean_squared_error(y_val_min, pred_min)))

print("\nMAX PRICE VALIDATION")
print("MAE :", mean_absolute_error(y_val_max, pred_max))
print("RMSE:", np.sqrt(mean_squared_error(y_val_max, pred_max)))

print("\nAverage Validation Prices:")
print("Min Avg:", y_val_min.mean())
print("Max Avg:", y_val_max.mean())


# =====================================================
# 6. STABLE RECURSIVE FORECAST FUNCTION
# =====================================================

def recursive_forecast(model_min, model_max, history_df, days=30):

    df_extended = history_df.copy()
    predictions = []

    for i in range(days):

        df_extended = create_features(df_extended)
        df_extended = df_extended.dropna()

        last_row = df_extended.iloc[-1]

        X_input = last_row[feature_cols].values.reshape(1, -1)

        pred_min = model_min.predict(X_input)[0]
        pred_max = model_max.predict(X_input)[0]

        prev_min = last_row['Carrot_Min_Price']
        prev_max = last_row['Carrot_Max_Price']

        # Limit daily movement to ±10%
        pred_min = np.clip(pred_min, prev_min * 0.90, prev_min * 1.10)
        pred_max = np.clip(pred_max, prev_max * 0.90, prev_max * 1.10)

        # Ensure logical consistency
        if pred_min > pred_max:
            pred_max = pred_min * 1.05

        next_date = last_row['Date'] + pd.Timedelta(days=1)

        new_row = {
            'Date': next_date,
            'Carrot_Min_Price': pred_min,
            'Carrot_Max_Price': pred_max,
            'Carrot_Avg_Price': (pred_min + pred_max) / 2
        }

        df_extended = pd.concat(
            [df_extended, pd.DataFrame([new_row])],
            ignore_index=True
        )

        predictions.append([next_date, pred_min, pred_max])

    return pd.DataFrame(
        predictions,
        columns=["Date", "Predicted_Min", "Predicted_Max"]
    )


# =====================================================
# 7. 30-DAY BACKTEST (FIXED VERSION)
# =====================================================

print("\nRunning 30-day recursive backtest...")

test_start = pd.to_datetime("2024-06-01")

history = df[df['Date'] < test_start].copy()

forecast_df = recursive_forecast(model_min, model_max, history, days=30)

# Get real values
future_real = df[
    (df['Date'] >= test_start) &
    (df['Date'] < test_start + pd.Timedelta(days=30))
][['Date', 'Carrot_Min_Price', 'Carrot_Max_Price']].copy()

# Merge safely by Date
comparison = pd.merge(
    forecast_df,
    future_real,
    on="Date",
    how="inner"
)

print("30-Day MAE (Min):",
      mean_absolute_error(
          comparison['Carrot_Min_Price'],
          comparison['Predicted_Min']
      ))

print("30-Day MAE (Max):",
      mean_absolute_error(
          comparison['Carrot_Max_Price'],
          comparison['Predicted_Max']
      ))


# =====================================================
# 8. SAVE MODELS
# =====================================================

os.makedirs("models", exist_ok=True)

joblib.dump(model_min, "models/carrot_min_model.pkl")
joblib.dump(model_max, "models/carrot_max_model.pkl")

print("\nModels saved successfully.")


MIN PRICE VALIDATION
MAE : 53.0169792175293
RMSE: 138.2453802347478

MAX PRICE VALIDATION
MAE : 68.79329681396484
RMSE: 189.49665028371874

Average Validation Prices:
Min Avg: 261.593567251462
Max Avg: 334.41520467836256

Running 30-day recursive backtest...
30-Day MAE (Min): 2.0
30-Day MAE (Max): 15.0

Models saved successfully.
